# Monitored XXZ domain wall

The initial four-spin state is $|\uparrow\uparrow\downarrow\downarrow\rangle$. The exchange term moves up-spin weight across the central bond while the Ising term changes neighboring-spin energies. Each period $\tau$, each site is independently selected for a projective $Z$ measurement with probability $p$. The observables are $\langle Z_i\rangle$ and $Q_R=\sum_{i>N/2}(1+\langle Z_i\rangle)/2$, the expected number of up spins in the initially down half. The dense channel averages over unread measurement outcomes; MPS trajectories retain individual outcomes. Execute cells in order with a Julia kernel and this project environment.


In [1]:
# Exact density-matrix channel: unitary motion plus unread local Z measurements.
using MonitoredXXZDomainWall, Random, LinearAlgebra
model = Model(4; J=0.7, delta=0.35, p=0.4, period=0.1)
exact = dense_channel(model, 5)
println("Exact final transfer: ", round(exact.transfer[end]; digits=8))
println("Exact final magnetization: ", round.(exact.magnetization[end,:]; digits=6))

Exact final transfer: 0.15066089
Exact final magnetization: [0.979001, 0.719677, -0.719677, -0.979001]


## Trajectory average and physical transfer

`sampled.mean_magnetization[k,i]` estimates the Pauli-$Z$ expectation at site `i` after period `k-1`. `sampled.mean_transfer[k]` is the right-half up-spin count $Q_R$, not an instantaneous current. The reported standard error covers sampling only; the gate split and bond truncation require separate refinement.


In [2]:
# Average 700 MPS measurement records; ± denotes the sampling standard error.
sampled = ensemble(model, 5, 700; maxdim=16, rng=MersenneTwister(23))
println("MPS final transfer: ", round(sampled.mean_transfer[end]; digits=8), " ± ", round(sampled.se_transfer[end]; digits=8))
println("Maximum magnetization difference: ", round(maximum(abs.(sampled.mean_magnetization-exact.magnetization)); digits=8))

MPS final transfer: 0.13256506 ± 0.01180447
Maximum magnetization difference: 0.03303403


## Closed-chain limit

Setting `p=0` removes measurements. Both the MPS and dense channel then describe unitary XXZ evolution. The final total $Z$ should remain zero because exchange moves up spins without creating them.


In [3]:
# At p=0, total Z is conserved and the evolution is purely unitary.
quiet = Model(4; J=0.7, delta=0.35, p=0, period=0.025)
unitary = trajectory(quiet, 16; maxdim=16, cutoff=0, rng=MersenneTwister(1))
unitary_exact = dense_channel(quiet, 16)
println("No-monitoring maximum magnetization difference: ", round(maximum(abs.(unitary.magnetization-unitary_exact.magnetization)); digits=8))
println("Final total Z: ", round(sum(unitary.magnetization[end,:]); digits=12))

No-monitoring maximum magnetization difference: 0.00015271
Final total Z: -0.0
